# Diamond Price Prediction and Market Segmentation
## Phase 4 - Exploratory Data Analysis (EDA)

This notebook performs:
- univariate analysis of key numeric variables
- categorical frequency analysis
- bivariate price analysis with `carat`, `cut`, `color`, `clarity`
- pairwise and correlation analysis
- markdown-ready observations for reporting

## Imports
Load the transformed interim dataset and the EDA helpers/plotting functions.

In [1]:
from pathlib import Path
import warnings

import pandas as pd

from src.utils.paths import find_project_root, resolve_project_path
from src.utils.config import load_project_configs
from src.utils.io import ensure_dir, save_text_file

from src.eda.univariate import build_univariate_summary
from src.eda.bivariate import build_price_category_summary, build_carat_price_summary
from src.eda.multivariate import build_correlation_matrix, build_pairplot_frame
from src.eda.eda_report import build_eda_report

from src.visualization.plot_eda import (
    plot_distribution,
    plot_countplot,
    plot_regplot,
    plot_avg_price_barplot,
    plot_pairplot,
    plot_correlation_heatmap,
    plot_numeric_boxplot_grid,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## Resolve project root and load the transformed dataset
EDA should use the skewness-treated interim dataset from the previous notebook.

In [2]:
project_root = find_project_root()
configs = load_project_configs(project_root)

eda_input_path = resolve_project_path(
    project_root,
    "data/interim/diamonds_skewness_treated.csv"
)

df = pd.read_csv(eda_input_path)
print("EDA input shape:", df.shape)
display(df.head())

EDA input shape: (53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,5.464589,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,5.464589,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,5.467317,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,5.486178,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,5.488840,4.34,4.35,2.75


## Define analysis columns

In [3]:
numeric_dist_cols = ["price", "carat", "x", "y", "z"]
categorical_count_cols = ["cut", "color", "clarity"]

eda_fig_dir = resolve_project_path(project_root, "figures/eda")
ensure_dir(eda_fig_dir)

WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda')

## Dataset snapshot
Review structure and summary statistics before plotting.

In [4]:
display(df[numeric_dist_cols].describe().T)

display(
    pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_count": df.isna().sum().values,
        "nunique": df.nunique().values,
    })
)

,count,mean,std,min,25%,50%,75%,max
price,53940.0,7.190010,0.846809,5.464589,6.40715,7.208076,7.884263,8.555295
carat,53940.0,0.792558,0.457089,0.200000,0.40000,0.700000,1.040000,2.000000
x,53940.0,5.731839,1.119016,3.730000,4.71000,5.700000,6.540000,9.285000
y,53940.0,5.733793,1.111132,3.680000,4.72000,5.710000,6.540000,9.270000
z,53940.0,3.539359,0.691047,1.215000,2.91000,3.530000,4.040000,5.735000


,column,dtype,missing_count,nunique
0,carat,float64,0,181
1,cut,object,0,5
2,color,object,0,7
3,clarity,object,0,8
4,depth,float64,0,184
5,table,float64,0,127
6,price,float64,0,9086
7,x,float64,0,533
8,y,float64,0,533
9,z,float64,0,351


## Plot numeric distributions
Create distribution plots for:
- `price`
- `carat`
- `x`
- `y`
- `z`

In [5]:
dist_paths = {
    "price": resolve_project_path(project_root, "figures/eda/dist_price.png"),
    "carat": resolve_project_path(project_root, "figures/eda/dist_carat.png"),
    "x": resolve_project_path(project_root, "figures/eda/dist_x.png"),
    "y": resolve_project_path(project_root, "figures/eda/dist_y.png"),
    "z": resolve_project_path(project_root, "figures/eda/dist_z.png"),
}

for col in numeric_dist_cols:
    plot_distribution(
        df=df,
        column=col,
        output_path=dist_paths[col],
    )

dist_paths

{'price': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/dist_price.png'),
 'carat': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/dist_carat.png'),
 'x': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/dist_x.png'),
 'y': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/dist_y.png'),
 'z': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/dist_z.png')}

## Plot categorical countplots
Create countplots for:
- `cut`
- `color`
- `clarity`

In [6]:
count_paths = {
    "cut": resolve_project_path(project_root, "figures/eda/count_cut.png"),
    "color": resolve_project_path(project_root, "figures/eda/count_color.png"),
    "clarity": resolve_project_path(project_root, "figures/eda/count_clarity.png"),
}

for col in categorical_count_cols:
    plot_countplot(
        df=df,
        column=col,
        output_path=count_paths[col],
    )

count_paths

{'cut': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/count_cut.png'),
 'color': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/count_color.png'),
 'clarity': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/figures/eda/count_clarity.png')}

## Analyze price variation with carat
Start with the direct numeric relationship between `carat` and `price`.

In [7]:
carat_price_summary = build_carat_price_summary(df)
display(carat_price_summary)

,metric,value
0,pearson_correlation_carat_price,0.921607
1,carat_mean,0.792558
2,price_mean,7.190010
3,sample_size,53940.000000


## Create carat vs price regression lineplot

In [8]:
carat_vs_price_path = resolve_project_path(
    project_root,
    "figures/eda/carat_vs_price_regplot.png"
)

plot_regplot(
    df=df,
    x_col="carat",
    y_col="price",
    output_path=carat_vs_price_path,
)

print("Saved:", carat_vs_price_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\eda\carat_vs_price_regplot.png


## Analyze average price by cut, color, and clarity

In [9]:
avg_price_by_cut = build_price_category_summary(df, category_col="cut")
avg_price_by_color = build_price_category_summary(df, category_col="color")
avg_price_by_clarity = build_price_category_summary(df, category_col="clarity")

display(avg_price_by_cut)
display(avg_price_by_color)
display(avg_price_by_clarity)

,cut,count,mean_price,median_price,min_price,max_price
0,Fair,1610,7.4568,7.4748,5.4941,8.5553
1,Premium,13791,7.3251,7.4492,5.4646,8.5553
2,Good,4906,7.2402,7.4125,5.4673,8.5553
3,Very Good,12082,7.2002,7.2918,5.4915,8.5553
4,Ideal,21551,7.0665,6.9656,5.4646,8.5553


,color,count,mean_price,median_price,min_price,max_price
0,J,2808,7.4886,7.6908,5.4888,8.5553
1,I,5422,7.3783,7.5834,5.4862,8.5553
2,H,8304,7.2987,7.5196,5.4941,8.5553
3,G,11292,7.1924,7.1494,5.5379,8.5553
4,F,9542,7.1721,7.1873,5.5072,8.5553
5,D,6775,7.0511,6.9788,5.5454,8.5553
6,E,9797,7.0189,6.9311,5.4646,8.5553


,clarity,count,mean_price,median_price,min_price,max_price
0,SI2,9194,7.5068,7.6578,5.4646,8.5553
1,I1,741,7.4080,7.4907,5.5150,8.5553
2,SI1,13065,7.2439,7.3461,5.4646,8.5553
3,VS2,12258,7.1690,7.0743,5.4862,8.5553
4,VS1,8171,7.1376,7.0535,5.4673,8.5553
5,VVS2,5066,6.9760,6.6871,5.4915,8.5553
6,IF,1790,6.8651,6.5188,5.5748,8.5553
7,VVS1,3655,6.7999,6.5292,5.4915,8.5553


## Save average price barplots

In [10]:
avg_cut_path = resolve_project_path(project_root, "figures/eda/avg_price_by_cut.png")
avg_color_path = resolve_project_path(project_root, "figures/eda/avg_price_by_color.png")
avg_clarity_path = resolve_project_path(project_root, "figures/eda/avg_price_by_clarity.png")

plot_avg_price_barplot(
    summary_df=avg_price_by_cut,
    category_col="cut",
    output_path=avg_cut_path,
)

plot_avg_price_barplot(
    summary_df=avg_price_by_color,
    category_col="color",
    output_path=avg_color_path,
)

plot_avg_price_barplot(
    summary_df=avg_price_by_clarity,
    category_col="clarity",
    output_path=avg_clarity_path,
)

print("Saved:", avg_cut_path)
print("Saved:", avg_color_path)
print("Saved:", avg_clarity_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\eda\avg_price_by_cut.png
Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\eda\avg_price_by_color.png
Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\eda\avg_price_by_clarity.png


## Pairplot
Create a pairplot for the main numeric variables to inspect pairwise structure.

In [11]:
pairplot_df = build_pairplot_frame(
    df=df,
    columns=["price", "carat", "x", "y", "z"],
    sample_size=3000,
    random_state=42,
)

pairplot_path = resolve_project_path(project_root, "figures/eda/pairplot.png")

plot_pairplot(
    df=pairplot_df,
    output_path=pairplot_path,
)

print("Saved:", pairplot_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\eda\pairplot.png


## Correlation heatmap
Inspect linear associations across the main numeric variables.

In [12]:
corr_df = build_correlation_matrix(
    df=df,
    columns=["price", "carat", "depth", "table", "x", "y", "z"],
)

heatmap_path = resolve_project_path(project_root, "figures/eda/correlation_heatmap.png")

plot_correlation_heatmap(
    corr_df=corr_df,
    output_path=heatmap_path,
)

display(corr_df)
print("Saved:", heatmap_path)

,price,carat,depth,table,x,y,z
price,1.000000,0.921607,0.002229,0.158900,0.955544,0.956109,0.951555
carat,0.921607,1.000000,0.026708,0.184356,0.983099,0.982231,0.981047
depth,0.002229,0.026708,1.000000,-0.295779,-0.025241,-0.028404,0.096163
table,0.158900,0.184356,-0.295779,1.000000,0.196074,0.189806,0.155780
x,0.955544,0.983099,-0.025241,0.196074,1.000000,0.998514,0.990593
y,0.956109,0.982231,-0.028404,0.189806,0.998514,1.000000,0.990336
z,0.951555,0.981047,0.096163,0.155780,0.990593,0.990336,1.000000


Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\eda\correlation_heatmap.png


## Numeric feature boxplots
Create a compact boxplot grid for numeric inspection.

In [13]:
boxplot_grid_path = resolve_project_path(
    project_root,
    "figures/eda/boxplots_numeric_features.png"
)

plot_numeric_boxplot_grid(
    df=df,
    columns=["price", "carat", "depth", "table", "x", "y", "z"],
    output_path=boxplot_grid_path,
)

print("Saved:", boxplot_grid_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\eda\boxplots_numeric_features.png


## Univariate summaries
Generate compact summaries for core numeric variables.

In [14]:
univariate_summary = build_univariate_summary(
    df=df,
    columns=["price", "carat", "depth", "table", "x", "y", "z"],
)

display(univariate_summary)

,column,count,missing_count,mean,median,std,min,q1,q3,max,skewness
0,price,53940,0,7.190010,7.208076,0.846809,5.464589,6.40715,7.884263,8.555295,0.008054
1,carat,53940,0,0.792558,0.700000,0.457089,0.200000,0.40000,1.040000,2.000000,0.899893
2,depth,53940,0,61.749405,61.800000,1.432621,43.000000,61.00000,62.500000,79.000000,-0.082294
3,table,53940,0,57.457184,57.000000,2.234491,43.000000,56.00000,59.000000,95.000000,0.796896
4,x,53940,0,5.731839,5.700000,1.119016,3.730000,4.71000,6.540000,9.285000,0.394179
5,y,53940,0,5.733793,5.710000,1.111132,3.680000,4.72000,6.540000,9.270000,0.389861
6,z,53940,0,3.539359,3.530000,0.691047,1.215000,2.91000,4.040000,5.735000,0.387198


## EDA observations
Write concise observations from the plots and summaries.

In [15]:
eda_observations = [
    "Price remains positively associated with carat, indicating carat is a dominant predictor of diamond value.",
    "The dimensional variables x, y, and z move closely with carat, suggesting strong multicollinearity among physical size variables.",
    "Cut, color, and clarity show meaningful price differences, confirming that categorical quality grades carry predictive signal.",
    "After outlier clipping and skewness treatment, the major numeric distributions are more stable for downstream regression modeling.",
    "The numeric variables are not purely normal even after treatment, so tree-based models will likely remain strong candidates alongside transformed linear baselines.",
]

for i, obs in enumerate(eda_observations, start=1):
    print(f"{i}. {obs}")

1. Price remains positively associated with carat, indicating carat is a dominant predictor of diamond value.
2. The dimensional variables x, y, and z move closely with carat, suggesting strong multicollinearity among physical size variables.
3. Cut, color, and clarity show meaningful price differences, confirming that categorical quality grades carry predictive signal.
4. After outlier clipping and skewness treatment, the major numeric distributions are more stable for downstream regression modeling.
5. The numeric variables are not purely normal even after treatment, so tree-based models will likely remain strong candidates alongside transformed linear baselines.


## Build markdown report
Save the EDA summary into `reports/eda_report.md`.

In [16]:
report_path = resolve_project_path(project_root, "reports/eda_report.md")

report_text = build_eda_report(
    df=df,
    univariate_summary_df=univariate_summary,
    carat_price_summary=carat_price_summary,
    avg_price_by_cut_df=avg_price_by_cut,
    avg_price_by_color_df=avg_price_by_color,
    avg_price_by_clarity_df=avg_price_by_clarity,
    corr_df=corr_df,
    observations=eda_observations,
    figure_paths={
        "dist_price": "figures/eda/dist_price.png",
        "dist_carat": "figures/eda/dist_carat.png",
        "dist_x": "figures/eda/dist_x.png",
        "dist_y": "figures/eda/dist_y.png",
        "dist_z": "figures/eda/dist_z.png",
        "count_cut": "figures/eda/count_cut.png",
        "count_color": "figures/eda/count_color.png",
        "count_clarity": "figures/eda/count_clarity.png",
        "carat_vs_price_regplot": "figures/eda/carat_vs_price_regplot.png",
        "avg_price_by_cut": "figures/eda/avg_price_by_cut.png",
        "avg_price_by_color": "figures/eda/avg_price_by_color.png",
        "avg_price_by_clarity": "figures/eda/avg_price_by_clarity.png",
        "pairplot": "figures/eda/pairplot.png",
        "correlation_heatmap": "figures/eda/correlation_heatmap.png",
        "boxplots_numeric_features": "figures/eda/boxplots_numeric_features.png",
    },
)

save_text_file(report_text, report_path)
print("Saved:", report_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\reports\eda_report.md


## Phase summary
Completed:
- numeric distribution plots
- categorical countplots
- price relationship analysis
- pairplot
- correlation heatmap
- average price barplots
- EDA observations
- markdown report